In [1]:
from PIL import Image
from IPython.display import display  # Optional, for local display if needed
import torch as th

from glide_text2im.download import load_checkpoint
from glide_text2im.model_creation import (
    create_model_and_diffusion,
    model_and_diffusion_defaults,
    model_and_diffusion_defaults_upsampler
)

c:\Users\Meet\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
has_cuda = th.cuda.is_available()
device = th.device('cpu' if not has_cuda else 'cuda')
print(f"Using device: {device}")

Using device: cpu


In [3]:
# Create base model
options = model_and_diffusion_defaults()
options['use_fp16'] = has_cuda
options['timestep_respacing'] = '100'  # 100 diffusion steps
model, diffusion = create_model_and_diffusion(**options)
model.eval()
if has_cuda:
    model.convert_to_fp16()
model.to(device)
model.load_state_dict(load_checkpoint('base', device))
print('total base parameters', sum(x.numel() for x in model.parameters()))

total base parameters 385030726


In [4]:
# Create upsampler model
options_up = model_and_diffusion_defaults_upsampler()
options_up['use_fp16'] = has_cuda
options_up['timestep_respacing'] = 'fast27'  # 27 diffusion steps
model_up, diffusion_up = create_model_and_diffusion(**options_up)
model_up.eval()
if has_cuda:
    model_up.convert_to_fp16()
model_up.to(device)
model_up.load_state_dict(load_checkpoint('upsample', device))
print('total upsampler parameters', sum(x.numel() for x in model_up.parameters()))

total upsampler parameters 398361286


In [5]:
# Define image save function
def save_images(batch):
    scaled = ((batch + 1) * 127.5).round().clamp(0, 255).to(th.uint8).cpu()
    reshaped = scaled.permute(2, 0, 3, 1).reshape([batch.shape[2], -1, 3])
    img = Image.fromarray(reshaped.numpy())
    img.save("generated_image.png")
    print("Image saved as generated_image.png")

In [6]:
prompt = "an oil painting of a corgi"
batch_size = 1
guidance_scale = 3.0
upsample_temp = 0.997  # Controls sharpness

In [7]:
# Create the text tokens
tokens = model.tokenizer.encode(prompt)
tokens, mask = model.tokenizer.padded_tokens_and_mask(tokens, options['text_ctx'])

# Create the classifier-free guidance tokens
full_batch_size = batch_size * 2
uncond_tokens, uncond_mask = model.tokenizer.padded_tokens_and_mask([], options['text_ctx'])

# Pack the tokens together
model_kwargs = dict(
    tokens=th.tensor([tokens] * batch_size + [uncond_tokens] * batch_size, device=device),
    mask=th.tensor([mask] * batch_size + [uncond_mask] * batch_size, dtype=th.bool, device=device),
)

# Create a classifier-free guidance sampling function
def model_fn(x_t, ts, **kwargs):
    half = x_t[: len(x_t) // 2]
    combined = th.cat([half, half], dim=0)
    model_out = model(combined, ts, **kwargs)
    eps, rest = model_out[:, :3], model_out[:, 3:]
    cond_eps, uncond_eps = th.split(eps, len(eps) // 2, dim=0)
    half_eps = uncond_eps + guidance_scale * (cond_eps - uncond_eps)
    eps = th.cat([half_eps, half_eps], dim=0)
    return th.cat([eps, rest], dim=1)

# Sample from the base model
model.del_cache()
samples = diffusion.p_sample_loop(
    model_fn,
    (full_batch_size, 3, options["image_size"], options["image_size"]),
    device=device,
    clip_denoised=True,
    progress=True,
    model_kwargs=model_kwargs,
    cond_fn=None,
)[:batch_size]
model.del_cache()

# Save the output
save_images(samples)

100%|██████████| 100/100 [17:07<00:00, 10.27s/it]


Image saved as generated_image.png


In [16]:
# Upsample the image (temporarily disabled due to error)
print("Samples shape before upsampling:", samples.shape)  # Debug: Check base model output shape
up_shape = (batch_size, 3, 256, 256)  # Hardcode to match GLIDE upsampler output
print("Up shape:", up_shape)  # Debug: Verify up_shape
print("Noise shape:", samples.shape)  # Debug: Verify noise shape
if samples.shape[2:] != (64, 64):  # Verify base model output
    raise ValueError("Base model output shape mismatch expected (1, 3, 64, 64)")

# Temporarily comment out upsampling to isolate the issue
"""
up_samples = diffusion_up.ddim_sample_loop(
    model_up,
    up_shape,
    noise=samples,  # Use original samples (64x64) as noise input
    device=device,
    clip_denoised=True,
    progress=True,
    model_kwargs={},
    cond_fn=None,
)[:batch_size]
"""

# Save the base image instead for now
save_images(samples)
print("Upsampling skipped due to error. Saved base image instead.")

# # Uncomment and fix the following line when upsampling is resolved
# # save_images(up_samples)

Samples shape before upsampling: torch.Size([1, 3, 64, 64])
Up shape: (1, 3, 256, 256)
Noise shape: torch.Size([1, 3, 64, 64])
Image saved as generated_image.png
Upsampling skipped due to error. Saved base image instead.
